# Support Vector Machines - Exercise 1

In this exercise, we'll be using support vector machines (SVMs) to build a spam classifier.  We'll start with SVMs on some simple 2D data sets to see how they work.  Then we'll do some pre-processing work on a set of raw emails and build a classifier on the processed emails using a SVM to determine if they are spam or not.

The first thing we're going to do is look at a simple 2-dimensional data set and see how a linear SVM works on the data set for varying values of C (similar to the regularization term in linear/logistic regression).  Let's load the data.
## Exercise 1
#### 1. Load libraries

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.svm import SVC
import matplotlib as mpl


mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

#### 2. Load data
Load the file *ejer_1_data1.mat*. Find the way for loading this kind of file. **scipy.io.loadmat**

In [2]:
# cargamos el archivo .mat (comúnmente usado en estos ejercicios de spam)
raw_data = loadmat('data/ex1data1.mat')

# extraemos las características (X) y las etiquetas (y)
X = raw_data['X']
y = raw_data['y'].ravel() # Aplanamos y para que sea un vector compatible con Scikit-Learn

# visualizamos los datos para entender la separación
def plot_data(X, y):
    plt.figure(figsize=(8, 6))
    # puntos donde y es 1 (positivo)
    plt.scatter(X[y == 1][:, 0], X[y == 1][:, 1], marker='+', c='black', label='Positivo')
    # puntos donde y es 0 (negativo)
    plt.scatter(X[y == 0][:, 0], X[y == 0][:, 1], marker='o', c='yellow', edgecolors='k', label='Negativo')
    plt.legend()

plot_data(X, y)
plt.title("Visualización de datos iniciales")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/ex1data1.mat'

#### 3. Create a DataFrame with the features and target

In [ ]:
# Caso A: C pequeño (C=1) -> Margen más ancho, permite algunos errores (Soft Margin)
svc1 = SVC(C=1, kernel='linear')
svc1.fit(X, y)
score1 = svc1.score(X, y)
print(f"Precisión con C=1: {score1*100:.2f}%")

# Caso B: C grande (C=100) -> Intenta clasificar todo correctamente, margen más estrecho
svc100 = SVC(C=100, kernel='linear')
svc100.fit(X, y)
score100 = svc100.score(X, y)
print(f"Precisión con C=100: {score100*100:.2f}%")

#### 4. Plot a scatterplot with the data

In [ ]:
# 4. Declarar SVC con hiperparámetros específicos
# C=100: Penalización alta por error
# gamma=10: Define qué tan lejos llega la influencia de un solo ejemplo (alto = influencia cercana)
# probability=True: Permite usar predict_proba() después
clf = SVC(C=100, gamma=10, kernel='rbf', probability=True)



Notice that there is one outlier positive example that sits apart from the others.  The classes are still linearly separable but it's a very tight fit.  We're going to train a linear support vector machine to learn the class boundary.

#### 5. LinearSVC
Declare a Linear SVC with the hyperparamenters:

```Python
LinearSVC(C=1, loss='hinge', max_iter=10000)
```

In [ ]:
# 5. Ajustar el clasificador y obtener la puntuación
clf.fit(X, y)
accuracy = clf.score(X, y)
print(f"Precisión final del modelo: {accuracy*100:.2f}%")

#### 6. Try the performance (score)
For the first experiment we'll use C=1 and see how it performs.

In [ ]:
# Creamos una malla (grid) para evaluar el modelo en todo el plano
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100),
                     np.linspace(y_min, y_max, 100))

# Predecimos la probabilidad para cada punto en la malla
# predict_proba devuelve [prob_clase_0, prob_clase_1]
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 0]
Z = Z.reshape(xx.shape)

# Graficamos
plt.figure(figsize=(10, 8))
# El mapa de colores 'viridis' o 'RdBu' es secuencial/divergente ideal para probabilidades
contour = plt.contourf(xx, yy, Z, cmap='viridis', alpha=0.8)
plt.colorbar(contour, label='Probabilidad de Clase 0')

# Superponemos los puntos originales
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='gray')
plt.title("Mapa de Probabilidades - Clasificador SVM RBF")
plt.xlabel("Característica 1")
plt.ylabel("Característica 2")
plt.show()

It appears that it mis-classified the outlier.

#### 7. Increase the value of C until you get a perfect classifier

In [ ]:
# 7. Incrementar el valor de C hasta obtener un clasificador perfecto
# Al subir C a 100, el SVM penaliza mucho más los errores (puntos mal clasificados)
svc_lineal_perfecto = LinearSVC(C=100, loss='hinge', max_iter=100000)
svc_lineal_perfecto.fit(X, y)
print(f"Precisión con C=100: {svc_lineal_perfecto.score(X, y)*100:.2f}%")


This time we got a perfect classification of the training data, however by increasing the value of C we've created a decision boundary that is no longer a natural fit for the data.  We can visualize this by looking at the confidence level for each class prediction, which is a function of the point's distance from the hyperplane.

#### 8. Plot Decission Function
Get the `decision_function()` output for the first model. Plot a scatterplot with X1, X2 and a range of colors based on `decision_function()`

In [ ]:

# 8. Graficar Decision Function (Modelo 1: C=1)
# decision_function() devuelve la distancia de cada punto al hiperplano
conf_c1 = svc_lineal.decision_function(X)
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=conf_c1, cmap='coolwarm')
plt.colorbar(label='Distancia al hiperplano (C=1)')
plt.title("Función de Decisión con C=1")
plt.show()


#### 9. Do the same with the second model

https://www.svm-tutorial.com/2015/06/svm-understanding-math-part-3/

In [ ]:

# 9. Hacer lo mismo con el segundo modelo (C=100)
conf_c100 = svc_lineal_perfecto.decision_function(X)
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=conf_c100, cmap='coolwarm')
plt.colorbar(label='Distancia al hiperplano (C=100)')
plt.title("Función de Decisión con C=100")
plt.show()

#### 1. Load the data `ejer_1_data2.mat`

In [ ]:
# 1. Cargar los datos 'ejer_1_data2.mat'
data2 = loadmat('data/ex1data2.mat')
X2 = data2['X']
y2 = data2['y'].ravel()


#### 2. Create a DataFrame with the features and target

In [ ]:

# 2. Crear un DataFrame con las características y el objetivo
df2 = pd.DataFrame(X2, columns=['X1', 'X2'])
df2['y'] = y2


#### 3. Plot a scatterplot with the data

In [ ]:

# 3. Graficar un scatterplot con los datos
plt.figure(figsize=(8, 6))
plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='viridis', marker='o', edgecolors='k')
plt.title("Datos No Lineales (Ejercicio 2)")
plt.show()


For this data set we'll build a support vector machine classifier using the built-in RBF kernel and examine its accuracy on the training data.  To visualize the decision boundary, this time we'll shade the points based on the predicted probability that the instance has a negative class label.  We'll see from the result that it gets most of them right.

#### 4. Declare a SVC with this hyperparameters
```Python
SVC(C=100, gamma=10, probability=True)
```


In [ ]:

# 4. Declarar un SVC con estos hiperparámetros
# Usamos C=100 para un ajuste fuerte y gamma=10 para capturar detalles locales
clf_rbf = SVC(C=100, gamma=10, kernel='rbf', probability=True)


#### 5. Fit the classifier and get the score

In [ ]:

# 5. Ajustar el clasificador y obtener el score
clf_rbf.fit(X2, y2)
print(f"Precisión Kernel RBF: {clf_rbf.score(X2, y2)*100:.2f}%")


#### 6. Plot the scatter plot and probability of predicting 0 with a [sequential color](https://matplotlib.org/3.1.1/tutorials/colors/colormaps.html)

In [ ]:

# 6. Graficar scatter plot y probabilidad de predecir 0
# Creamos una rejilla para pintar el fondo de color según la probabilidad
h = .02
x_min, x_max = X2[:, 0].min() - 0.1, X2[:, 0].max() + 0.1
y_min, y_max = X2[:, 1].min() - 0.1, X2[:, 1].max() + 0.1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Obtenemos la probabilidad de la clase 0
prob = clf_rbf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 0]
prob = prob.reshape(xx.shape)

plt.figure(figsize=(10, 8))
# Usamos un mapa de color secuencial 'YlGnBu' para la probabilidad
plt.contourf(xx, yy, prob, cmap='YlGnBu', alpha=0.8)
plt.colorbar(label='Probabilidad de Clase 0 (Negativo)')
# Dibujamos los puntos encima
plt.scatter(X2[:, 0], X2[:, 1], c=y2, edgecolors='k', cmap='gray')
plt.title("Probabilidades de Predicción con Kernel RBF")
plt.show()

In [ ]:
'''
C=1 vs C=100: En el punto 7, al aumentar C, notarás que el modelo ya no ignora el punto atípico
(outlier). Se vuelve más preciso en el entrenamiento, pero el límite de decisión puede volverse
menos "natural".

decision_function(): En los puntos 8 y 9, los colores representan qué tan lejos está un punto
de la frontera. Los colores más intensos indican una mayor confianza en la clasificación.

Sequential Color en RBF: En el ejercicio final, se utiliza un mapa de color para visualizar
cómo la probabilidad cambia suavemente a medida que nos alejamos de los grupos de puntos.
'''